# Task 1 – Netflix Data Cleaning & Preparation
**Auspify Technologies – Data Analysis Using Python Internship**

**Goal:** Prepare and organize the Netflix dataset for business analytics and reporting.

**Workflow:** Import → Missing values → Duplicates & formatting → Standardize Country/Rating/Type → Export

## Step 1: Import the dataset using Python and Pandas

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Dataset.csv')
raw_rows = len(df)
print('Shape:', df.shape)
df.head()

Shape: (8790, 10)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,8790,8790,s1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,8790,2,Movie,6126,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,8790,8787,9-Feb,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
director,8790,4528,Not Given,2588,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,8790,86,United States,3240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date_added,8790,1713,1/1/2020,110,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,8790.0,NaN,NaN,NaN,2014.183163,8.825466,1925.0,2013.0,2017.0,2019.0,2021.0
rating,8790,14,TV-MA,3205,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,8790,220,1 Season,1791,NaN,NaN,NaN,NaN,NaN,NaN,NaN
listed_in,8790,513,"Dramas, International Movies",362,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 2: Identify and handle missing values
First check for true NaN values, then look for *hidden* missing values (placeholder text such as `Not Given`).

In [4]:
print('True NaN values per column:')
print(df.isnull().sum())

print('\nHidden missing values (placeholder "Not Given"):')
for col in ['director', 'country', 'rating', 'type']:
    print(f'{col:10s}', (df[col].str.strip().str.lower() == 'not given').sum())

True NaN values per column:
show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

Hidden missing values (placeholder "Not Given"):
director   2588
country    287
rating     0
type       0


**Finding:** There are no true NaNs, but `director` (2,588 rows) and `country` (287 rows) use `Not Given` as a placeholder. Pandas does not see these as missing, so we convert them to real `NaN` first, then decide how to handle each one:
- `director` → fill with **'Unknown'** (too many rows to drop; director is not needed for most analyses)
- `country` → fill with **'Unknown'** (keeps the title in type/year/rating analysis; Task 3 can filter it out)

In [5]:
# Convert placeholder text into real missing values
placeholders = ['not given', 'n/a', 'na', 'none', 'null', '']
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()
    df.loc[df[col].str.lower().isin(placeholders), col] = np.nan

print(df.isnull().sum())

show_id            0
type               0
title              0
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64


C:\Users\Admin\AppData\Local\Temp\ipykernel_15964\2023893221.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


In [6]:
# Handle the missing values
df['director'] = df['director'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')

print('Remaining missing values:', df.isnull().sum().sum())

Remaining missing values: 0


## Step 3: Remove duplicate records and formatting inconsistencies

In [7]:
# Exact duplicates
print('Exact duplicate rows:', df.duplicated().sum())

# Same content but a different show_id (hidden duplicates)
content_cols = [c for c in df.columns if c != 'show_id']
print('Duplicates ignoring show_id:', df.duplicated(subset=content_cols).sum())
df[df.duplicated(subset=content_cols, keep=False)].sort_values('title')

Exact duplicate rows: 0
Duplicates ignoring show_id: 4


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
2925,s3963,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"
3285,s4523,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
393,s3997,TV Show,9-Feb,Unknown,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
537,s5965,TV Show,9-Feb,Unknown,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
2590,s3372,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,TV-MA,106 min,"Dramas, International Movies, Thrillers"
4718,s6530,Movie,Consequences,Ozan Açıktan,Turkey,10/25/2019,2014,TV-MA,106 min,"Dramas, International Movies, Thrillers"


In [8]:
df = df.drop_duplicates(subset=content_cols, keep='first').reset_index(drop=True)
print('Rows after removing duplicates:', len(df))

Rows after removing duplicates: 8786


### Fix formatting inconsistencies
- `date_added` is stored as text → convert to a real datetime and extract year / month
- `duration` mixes minutes and seasons in one text column → split into a numeric value and a unit

In [9]:
# date_added -> datetime
df['date_added'] = pd.to_datetime(df['date_added'], format='%m/%d/%Y', errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month_name()
print('Unparsed dates:', df['date_added'].isnull().sum())

# duration -> number + unit
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(int)
df['duration_unit'] = df['duration'].str.extract(r'\d+\s*(\w+)')[0].str.lower().replace({'seasons': 'season'})
df[['duration', 'duration_value', 'duration_unit']].head()

Unparsed dates: 0


,duration,duration_value,duration_unit
0,90 min,90,min
1,1 Season,1,season
2,1 Season,1,season
3,91 min,91,min
4,125 min,125,min


In [10]:
# Text columns: remove extra whitespace
for col in ['title', 'director', 'listed_in']:
    df[col] = df[col].str.strip().str.replace(r'\s+', ' ', regex=True)

# Sanity check: release_year range
print(df['release_year'].min(), '-', df['release_year'].max())

1925 - 2021


## Step 4: Standardize columns – Country, Rating and Type

In [11]:
# TYPE: consistent labels
df['type'] = df['type'].str.strip().str.title().replace({'Tv Show': 'TV Show'})
print(df['type'].value_counts())

type
Movie      6123
TV Show    2663
Name: count, dtype: int64


In [12]:
# COUNTRY: trim, title-case, fix common variants
df['country'] = df['country'].str.strip().str.replace(r'\s+', ' ', regex=True)
country_fix = {'USA': 'United States', 'US': 'United States', 'UK': 'United Kingdom', 'U.S.': 'United States'}
df['country'] = df['country'].replace(country_fix)
print('Unique countries:', df['country'].nunique())
df['country'].value_counts().head(10)

Unique countries: 86


country
United States     3240
India             1056
United Kingdom     638
Pakistan           420
Unknown            287
Canada             271
Japan              259
South Korea        214
France             213
Spain              182
Name: count, dtype: int64

In [13]:
# RATING: uppercase/trim, then group into audience categories
df['rating'] = df['rating'].str.strip().str.upper()
print(df['rating'].value_counts())

rating_group = {
    'TV-Y': 'Kids', 'TV-G': 'Kids', 'G': 'Kids', 'TV-Y7': 'Older Kids', 'TV-Y7-FV': 'Older Kids',
    'PG': 'Older Kids', 'TV-PG': 'Older Kids',
    'TV-14': 'Teens', 'PG-13': 'Teens',
    'TV-MA': 'Adults', 'R': 'Adults', 'NC-17': 'Adults',
    'NR': 'Unrated', 'UR': 'Unrated'
}
df['rating_group'] = df['rating'].map(rating_group)
print('\nUnmapped ratings:', df['rating_group'].isnull().sum())
df['rating_group'].value_counts()

rating
TV-MA       3204
TV-14       2155
TV-PG        861
R            798
PG-13        490
TV-Y7        333
TV-Y         306
PG           287
TV-G         220
NR            79
G             41
TV-Y7-FV       6
NC-17          3
UR             3
Name: count, dtype: int64

Unmapped ratings: 0


rating_group
Adults        4005
Teens         2645
Older Kids    1487
Kids           567
Unrated         82
Name: count, dtype: int64

## Final quality check

In [14]:
print('Rows: raw =', raw_rows, '| cleaned =', len(df))
print('Rows removed (duplicates):', raw_rows - len(df))
print('Missing values:', df.isnull().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
print()
df.info()

Rows: raw = 8790 | cleaned = 8786
Rows removed (duplicates): 4
Missing values: 0
Duplicate rows: 0

<class 'pandas.DataFrame'>
RangeIndex: 8786 entries, 0 to 8785
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   show_id         8786 non-null   str           
 1   type            8786 non-null   str           
 2   title           8786 non-null   str           
 3   director        8786 non-null   str           
 4   country         8786 non-null   str           
 5   date_added      8786 non-null   datetime64[us]
 6   release_year    8786 non-null   int64         
 7   rating          8786 non-null   str           
 8   duration        8786 non-null   str           
 9   listed_in       8786 non-null   str           
 10  year_added      8786 non-null   int32         
 11  month_added     8786 non-null   str           
 12  duration_value  8786 non-null   int64         
 13  duration_unit   878

In [15]:
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,year_added,month_added,duration_value,duration_unit,rating_group
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries,2021,September,90,min,Teens
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,September,1,season,Adults
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",2021,September,1,season,Adults
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies",2021,September,91,min,Older Kids
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies",2021,September,125,min,Adults


## Step 5: Export the cleaned dataset for analysis

In [16]:
df.to_csv('netflix_cleaned.csv', index=False)
print('Saved netflix_cleaned.csv ->', df.shape)

Saved netflix_cleaned.csv -> (8786, 15)


## Summary of cleaning actions
| # | Issue found | Action taken |
|---|---|---|
| 1 | `Not Given` placeholders in `director` (2,588) and `country` (287) | Converted to NaN, then filled with `Unknown` |
| 2 | 4 duplicate titles (same content, different `show_id`; 'Consequences' was hidden by extra whitespace) | Trimmed text, then removed duplicates, kept first |
| 3 | `date_added` stored as text | Converted to datetime; added `year_added`, `month_added` |
| 4 | `duration` mixed minutes/seasons in one text field | Split into `duration_value` and `duration_unit` |
| 5 | Inconsistent text/whitespace | Stripped and normalised |
| 6 | Type / Country / Rating standardised | Consistent labels; added `rating_group` (Kids / Older Kids / Teens / Adults / Unrated) |

The cleaned file `netflix_cleaned.csv` is the input for Tasks 2–6.